# 09 Weekly Challenge

전체 프로세스: Qwen Model → LoRA/QLoRA → PTQ → GGUF, Llama.cpp

## 진행 순서
- Phase 0. Baseline 확보
- Phase 1. Fine-Tuning: LoRA vs QLoRA
- Phase 2. Post-Training Quantization (PTQ)
- Phase 3. GGUF 변환 및 Llama.cpp 추론
- 최종 정리 (표 1, 표 2)

## Phase 0. Baseline 확보

- 0-1. 환경 설정 및 라이브러리 import
- 0-2. Qwen 원본 모델 로드 (bfloat16)
- 0-3. 샘플 prompt 추론 테스트
- 0-4. Baseline metrics 기록 (perplexity, memory, latency)
- 0-5. [Empty Cache] 원본 모델 메모리 해제

In [21]:
# 0-1. 환경 설정 및 라이브러리 import
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")

# 전체 Phase에 걸쳐 metrics를 누적할 딕셔너리 (표 1 원본 데이터)
performance_metrics_by_phase = {}

device: cuda


In [22]:
# 0-2. Qwen 원본 모델 로드 (bfloat16)
model_name_or_path = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

original_qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    dtype=torch.bfloat16
).to(device)

original_qwen_model.eval()
print(f"model loaded: {model_name_or_path}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [23]:
# 0-3. 샘플 prompt 추론 테스트
sample_prompt = "다음 주 화요일 오후 3시에 회의 일정을 잡아줘."

input_token_ids = tokenizer(sample_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated_token_ids = original_qwen_model.generate(
        **input_token_ids,
        max_new_tokens=100,
        do_sample=False
    )

generated_text = tokenizer.decode(generated_token_ids[0], skip_special_tokens=True)
print(generated_text)

다음 주 화요일 오후 3시에 회의 일정을 잡아줘. 그리고 그날 저녁에는 어떤 음식을 먹어야 할지 알려주세요.
주말에 회의를 가질 예정입니다. 다음 주 화요일 오후 3시에 회의를 진행할 계획이니, 그 날의 일정을 잡아주시기 바랍니다.

1. 오전 9시: 회의 준비
2. 오후 10시: 회의 시작

그날 저녁은 무엇을 �


In [24]:
# 0-4. Baseline metrics 기록 (perplexity, memory, latency)
def measure_memory_usage_in_megabytes(model):
    total_parameter_bytes = sum(
        parameter.element_size() * parameter.numel()
        for parameter in model.parameters()
    )
    return total_parameter_bytes / (1024 ** 2)


def measure_inference_latency_in_seconds(model, tokenizer, prompt_text, device, number_of_runs=5):
    input_token_ids = tokenizer(prompt_text, return_tensors="pt").to(device)

    # warm-up (초기 실행 오버헤드 제외)
    with torch.no_grad():
        model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)

    elapsed_time_list = []
    for _ in range(number_of_runs):
        start_time = time.time()
        with torch.no_grad():
            model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)
        elapsed_time_list.append(time.time() - start_time)

    return sum(elapsed_time_list) / len(elapsed_time_list)


def measure_perplexity(model, tokenizer, evaluation_text_list, device):
    total_negative_log_likelihood = 0.0
    total_token_count = 0

    for evaluation_text in evaluation_text_list:
        input_token_ids = tokenizer(evaluation_text, return_tensors="pt").to(device)
        with torch.no_grad():
            model_output = model(**input_token_ids, labels=input_token_ids["input_ids"])

        token_count = input_token_ids["input_ids"].size(1)
        total_negative_log_likelihood += model_output.loss.item() * token_count
        total_token_count += token_count

    average_negative_log_likelihood = total_negative_log_likelihood / total_token_count
    return torch.exp(torch.tensor(average_negative_log_likelihood)).item()

In [25]:
evaluation_text_list = [
    "다음 주 화요일 오후 3시에 회의 일정을 잡아줘.",
    "이번 주 금요일에 잡힌 일정이 있는지 확인해줘.",
    "내일 오전 10시에 팀 미팅 일정을 추가해줘.",
    "다음 달 첫째 주에 워크숍 일정을 등록해줘.",
    "오늘 오후 6시 저녁 약속을 캘린더에 저장해줘.",
    "이번 주 수요일 일정을 모두 삭제해줘.",
    "다음 주 월요일부터 금요일까지 매일 아침 회의를 잡아줘.",
    "이번 달 마지막 주에 휴가 일정을 등록해줘.",
    "내일 오후 2시로 잡힌 미팅을 오후 4시로 변경해줘.",
    "이번 주말에 잡힌 일정이 있으면 알려줘.",
    "다음 주 목요일 점심 약속을 취소해줘.",
    "매주 화요일 오전 9시에 반복 일정을 등록해줘.",
    "이번 주 일정 중 가장 빠른 일정이 무엇인지 알려줘.",
    "다음 주 출장 일정을 캘린더에 추가해줘.",
    "오늘 저녁에 잡힌 약속 시간을 확인해줘.",
    "이번 달 중 비어있는 날짜를 찾아줘.",
    "다음 주 화요일 회의를 다른 요일로 옮겨줘.",
    "이번 주 금요일 오후 일정을 모두 보여줘.",
    "내일 오전 회의 참석자 명단을 확인해줘.",
    "다음 주에 예정된 모든 일정을 요약해줘.",
]

performance_metrics_by_phase["baseline"] = {
    "memory_mb": measure_memory_usage_in_megabytes(original_qwen_model),
    "latency_sec": measure_inference_latency_in_seconds(
        original_qwen_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        original_qwen_model, tokenizer, evaluation_text_list, device
    ),
}

print(performance_metrics_by_phase["baseline"])

{'memory_mb': 2944.4013671875, 'latency_sec': 2.1623411655426024, 'perplexity': 13.661505699157715}


In [26]:
# 0-5. [Empty Cache] 원본 모델 메모리 해제
def empty_device_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    # 둘 다 없으면 (cpu) 아무것도 하지 않음

gc.collect()
empty_device_cache()

### Phase 0 실행 현황

- 실행 환경: VSCode ↔ Colab 원격 연결, GPU 런타임 (CUDA, T4)
- device 분기 및 empty cache 함수를 CUDA/MPS/CPU 대응 버전으로 수정 후 정상 동작 확인
- Baseline 확정 (Qwen2.5-1.5B-Instruct, bfloat16 기준)
  - memory_mb: 2944.40
  - latency_sec: 2.10 (50 tokens 생성 기준)
  - perplexity: 13.66 (evaluation 문장 20개 기준)
- Phase 1(LoRA/QLoRA) 진행을 위한 baseline 값으로 확정

---

## Phase 1. Fine-Tuning: LoRA vs QLoRA

- 1-1. Dataset 준비 (DaySync 도메인 데이터)
- 1-2. LoRA
- 1-3. QLoRA
- 1-4. LoRA vs QLoRA 비교 및 선택

In [27]:
# 1-1. Dataset 
# 1-1-1. Train / Validation Split (19 / 5)
instruction_response_pair_list = [
    {
        "instruction": "다음 주 화요일 오후 3시에 회의 일정을 잡아줘.",
        "response": "다음 주 화요일 오후 3시에 회의 일정을 등록했습니다."
    },
    {
        "instruction": "이번 주 금요일에 잡힌 일정이 있는지 확인해줘.",
        "response": "이번 주 금요일에 등록된 일정은 오후 2시 팀 회의입니다."
    },
    {
        "instruction": "내일 오전 10시에 팀 미팅 일정을 추가해줘.",
        "response": "내일 오전 10시에 팀 미팅 일정을 추가했습니다."
    },
    {
        "instruction": "다음 달 첫째 주에 워크숍 일정을 등록해줘.",
        "response": "다음 달 첫째 주 월요일에 워크숍 일정을 등록했습니다."
    },
    {
        "instruction": "오늘 오후 6시 저녁 약속을 캘린더에 저장해줘.",
        "response": "오늘 오후 6시 저녁 약속을 캘린더에 저장했습니다."
    },
    {
        "instruction": "이번 주 수요일 일정을 모두 삭제해줘.",
        "response": "이번 주 수요일에 등록된 일정을 모두 삭제했습니다."
    },
    {
        "instruction": "다음 주 월요일부터 금요일까지 매일 아침 회의를 잡아줘.",
        "response": "다음 주 월요일부터 금요일까지 매일 아침 9시에 회의 일정을 등록했습니다."
    },
    {
        "instruction": "이번 달 마지막 주에 휴가 일정을 등록해줘.",
        "response": "이번 달 마지막 주 월요일부터 금요일까지 휴가 일정을 등록했습니다."
    },
    {
        "instruction": "내일 오후 2시로 잡힌 미팅을 오후 4시로 변경해줘.",
        "response": "내일 미팅 일정을 오후 2시에서 오후 4시로 변경했습니다."
    },
    {
        "instruction": "이번 주말에 잡힌 일정이 있으면 알려줘.",
        "response": "이번 주말에는 등록된 일정이 없습니다."
    },
    {
        "instruction": "다음 주 목요일 점심 약속을 취소해줘.",
        "response": "다음 주 목요일 점심 약속을 취소했습니다."
    },
    {
        "instruction": "매주 화요일 오전 9시에 반복 일정을 등록해줘.",
        "response": "매주 화요일 오전 9시 반복 일정을 등록했습니다."
    },
    {
        "instruction": "이번 주 일정 중 가장 빠른 일정이 무엇인지 알려줘.",
        "response": "이번 주 가장 빠른 일정은 월요일 오전 9시 회의입니다."
    },
    {
        "instruction": "다음 주 출장 일정을 캘린더에 추가해줘.",
        "response": "다음 주 수요일부터 목요일까지 출장 일정을 캘린더에 추가했습니다."
    },
    {
        "instruction": "오늘 저녁에 잡힌 약속 시간을 확인해줘.",
        "response": "오늘 저녁 약속은 오후 7시입니다."
    },
    {
        "instruction": "이번 달 중 비어있는 날짜를 찾아줘.",
        "response": "이번 달 중 15일과 22일이 비어있습니다."
    },
    {
        "instruction": "다음 주 화요일 회의를 다른 요일로 옮겨줘.",
        "response": "다음 주 화요일 회의를 수요일로 옮겼습니다."
    },
    {
        "instruction": "이번 주 금요일 오후 일정을 모두 보여줘.",
        "response": "이번 주 금요일 오후 일정은 3시 회의, 5시 보고입니다."
    },
    {
        "instruction": "내일 오전 회의 참석자 명단을 확인해줘.",
        "response": "내일 오전 회의 참석자는 총 5명입니다."
    },
    {
        "instruction": "다음 주에 예정된 모든 일정을 요약해줘.",
        "response": "다음 주에는 회의 3건, 출장 1건, 워크숍 1건이 예정되어 있습니다."
    },
    {
        "instruction": "이번 주 화요일 회의 시간을 알려줘.",
        "response": "이번 주 화요일 회의는 오후 3시입니다."
    },
    {
        "instruction": "다음 주 수요일 오전 일정을 비워줘.",
        "response": "다음 주 수요일 오전 일정을 모두 비웠습니다."
    },
    {
        "instruction": "오늘 일정 중 취소된 항목이 있는지 알려줘.",
        "response": "오늘 일정 중 취소된 항목은 없습니다."
    },
    {
        "instruction": "이번 주 목요일에 새로운 회의를 추가해줘.",
        "response": "이번 주 목요일 오후 1시에 새로운 회의를 추가했습니다."
    }
]

print(f"total pairs: {len(instruction_response_pair_list)}")

# 1-1-2. Prompt Template 구성 (Qwen Instruct format)
train_pair_list = instruction_response_pair_list[:19]
validation_pair_list = instruction_response_pair_list[19:]

print(f"train: {len(train_pair_list)}, validation: {len(validation_pair_list)}")

# 1-1-3. Prompt Template 구성 (Qwen Instruct format)
"""
Qwen2.5-Instruct는 ChatML 형식(chat template)을 사용하므로, tokenizer.apply_chat_template으로 변환.
"""
def build_chat_formatted_text(instruction_text, response_text, tokenizer):
    message_list = [
        {"role": "user", "content": instruction_text},
        {"role": "assistant", "content": response_text}
    ]
    return tokenizer.apply_chat_template(message_list, tokenize=False)

formatted_train_text_list = [
    build_chat_formatted_text(pair["instruction"], pair["response"], tokenizer)
    for pair in train_pair_list
]

print(formatted_train_text_list[0])

# 1-1-4. Dataset 객체 변환 (Hugging Face datasets)
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": formatted_train_text_list})

formatted_validation_text_list = [
    build_chat_formatted_text(pair["instruction"], pair["response"], tokenizer)
    for pair in validation_pair_list
]
validation_dataset = Dataset.from_dict({"text": formatted_validation_text_list})

print(train_dataset)
print(validation_dataset)

total pairs: 24
train: 19, validation: 5
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
다음 주 화요일 오후 3시에 회의 일정을 잡아줘.<|im_end|>
<|im_start|>assistant
다음 주 화요일 오후 3시에 회의 일정을 등록했습니다.<|im_end|>

Dataset({
    features: ['text'],
    num_rows: 19
})
Dataset({
    features: ['text'],
    num_rows: 5
})


In [28]:
# 1-2. LoRA
## 1-2-1. 기반 모델 로드 (bfloat16)
from transformers import AutoModelForCausalLM

lora_base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    dtype=torch.bfloat16
).to(device)

print(f"lora base model loaded on: {next(lora_base_model.parameters()).device}")

## 1-2-2. LoRA Config 설정 (rank, alpha, target_modules)
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.print_trainable_parameters()

## 1-2-3. 학습 실행
def tokenize_function(example_batch):
    tokenized_output = tokenizer(
        example_batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    tokenized_output["labels"] = tokenized_output["input_ids"].copy()
    return tokenized_output


tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)

from transformers import TrainingArguments, Trainer

lora_training_arguments = TrainingArguments(
    output_dir="./lora_checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_arguments,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset
)

lora_train_result = lora_trainer.train()

## 1-2-4. LoRA metrics 기록
lora_evaluation_result = lora_trainer.evaluate()

performance_metrics_by_phase["lora"] = {
    "memory_mb": measure_memory_usage_in_megabytes(lora_model),
    "latency_sec": measure_inference_latency_in_seconds(
        lora_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        lora_model, tokenizer, evaluation_text_list, device
    ),
    "train_loss": lora_train_result.training_loss,
    "eval_loss": lora_evaluation_result["eval_loss"],
}

print(performance_metrics_by_phase["lora"])

## 1-2-5. [Empty Cache]
del lora_base_model, lora_model, lora_trainer
gc.collect()
empty_device_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

lora base model loaded on: cuda:0


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# 1-3. QLoRA
## 1-3-1. 기반 모델 로드 (4-bit NF4, BitsAndBytesConfig)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    quantization_config=bnb_config,
    dtype=torch.bfloat16
)

print(f"qlora base model loaded on: {next(qlora_base_model.parameters()).device}")

## 1-3-2. LoRA Config 설정 (1-2-2와 동일 설정값 재사용)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

qlora_base_model = prepare_model_for_kbit_training(qlora_base_model)

qlora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

qlora_model = get_peft_model(qlora_base_model, qlora_config)
qlora_model.print_trainable_parameters()

## 1-3-3. 학습 실행
from transformers import TrainingArguments, Trainer

qlora_training_arguments = TrainingArguments(
    output_dir="./qlora_checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

qlora_trainer = Trainer(
    model=qlora_model,
    args=qlora_training_arguments,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset
)

qlora_train_result = qlora_trainer.train()

## 1-3-4. QLoRA metrics 기록
qlora_evaluation_result = qlora_trainer.evaluate()

performance_metrics_by_phase["qlora"] = {
    "memory_mb": measure_memory_usage_in_megabytes(qlora_model),
    "latency_sec": measure_inference_latency_in_seconds(
        qlora_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        qlora_model, tokenizer, evaluation_text_list, device
    ),
    "train_loss": qlora_train_result.training_loss,
    "eval_loss": qlora_evaluation_result["eval_loss"],
}

print(performance_metrics_by_phase["qlora"])

## 1-3-5. [Empty Cache]
del qlora_base_model, qlora_model, qlora_trainer
gc.collect()
empty_device_cache()

In [ ]:
# 1-4. LoRA vs QLoRA 비교 및 선택
## 1-4-1. 비교 표 작성 (perplexity, exact match, 메모리, latency)
import pandas as pd

comparison_columns_list = ["memory_mb", "latency_sec", "perplexity", "train_loss", "eval_loss"]

comparison_dataframe = pd.DataFrame(
    {
        "baseline": performance_metrics_by_phase["baseline"],
        "lora": performance_metrics_by_phase["lora"],
        "qlora": performance_metrics_by_phase["qlora"],
    }
).T

print(comparison_dataframe)

## 1-4-2. 다음 Phase로 전달할 모델 선택 및 근거 기록
selection_criteria_summary = f"""
선택 기준:
1. perplexity: baseline({performance_metrics_by_phase['baseline']['perplexity']:.2f}) 대비
   LoRA({performance_metrics_by_phase['lora']['perplexity']:.2f}), 
   QLoRA({performance_metrics_by_phase['qlora']['perplexity']:.2f}) 중 더 낮은 쪽이 우수
2. memory_mb: QLoRA가 4-bit 기반이므로 더 낮을 것으로 예상 — 실제 수치로 검증 필요
3. eval_loss: 과적합 여부 판단 (train_loss 대비 eval_loss 격차가 크면 과적합 의심)
4. latency_sec: 추론 속도 비교
"""
print(selection_criteria_summary)
# 실제 수치 확인 후 아래 변수에 선택 결과와 근거를 기록
selected_model_name = None  # "lora" 또는 "qlora"
selection_reason = None     # 위 4가지 기준 중 어떤 근거로 선택했는지 서술

performance_metrics_by_phase["selected_for_next_phase"] = {
    "model": selected_model_name,
    "reason": selection_reason
}

## Phase 2. Post-Training Quantization (PTQ)

### 2-1. 양자화 방식 결정
#### 2-1-1. Weight-only vs Full Quantization 선택 및 근거
#### 2-1-2. GPTQ vs AWQ 선택 및 근거
#### 2-1-3. Static vs Dynamic Quantization 선택 및 근거 (Activation Quantization 포함 시)
### 2-2. Calibration Data 준비 (Static 선택 시)
### 2-3. 양자화 실행
### 2-4. PTQ metrics 기록 (Baseline 대비, 직전 Phase 대비)
### 2-5. [Empty Cache]

## Phase 3. GGUF 변환 및 Llama.cpp 추론

### 3-1. Adapter Merge 여부 결정 및 실행
### 3-2. GGUF 변환
### 3-3. Llama.cpp 로드 및 추론 테스트
### 3-4. Phase 2 결과와 출력 일치 여부 확인 (변환 손실 검증)
### 3-5. GGUF metrics 기록

## 최종 정리

### 표 1. 원본 Qwen 대비 누적 비교 (Baseline / Fine-Tuned / PTQ / GGUF)
### 표 2. 단계별 순수 변화량 (표 1의 인접 행 차이, 파생 계산)
### 결론 및 mentoring 검증 항목 정리